# Testing and debugging the dataset creation

In [1]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
import time
import sys
sys.path.append("..")
import importlib
import scripts.ingest

importlib.reload(scripts.ingest)

from scripts.ingest import (
    processed_date,
    scrape_archive_metadata,
    selection_logic,
    extract_slug,
    scrape_or_load_edition,
    filter_and_clean_parts,
    url_headline_extract,
    process_edition,
    append_and_store_edition,
    append_and_store_quiz,
    check_existence_process_append_store
)

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

client = Firecrawl(api_key=api_key)

data_path = "../data"

with open(f"{data_path}/archive.json", "r") as f:
    archive = json.load(f)

In [2]:
full_archive_metadata = scrape_archive_metadata(client)

In [3]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

data_path = "../data"

index = 421
edition_metadata = full_archive_metadata[index]

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
print(date)
edition_metadata

2025-06-26


{'_id': 'c2JB5yBiFSSXjqY6QpTPqE',
 'date': '2025-06-26T06:00:00-04:00',
 'previewText': 'What the Iran conflict means for gas prices...',
 'slug': 'wexit',
 'subjectLine': 'Wexit',
 'title': None,
 'websitePreviewText': None}

In [4]:
selection_logic(date)

True

In [5]:
slug = extract_slug(edition_metadata, date)
print(slug)

wexit


In [7]:
edition_metadata = full_archive_metadata[index]
edition_id = edition_metadata["_id"]
iso_date = date.isoformat()

scraped_edition = scrape_or_load_edition(client, slug, edition_id, date, archive)
scraped_edition

Edition for 2025-06-26 found


"[Skip to main content](https://www.morningbrew.com/issues/wexit#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Wexit\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| What the Iran conflict means for gas prices... |\n\n[![](https://media.sailthru.com/5z8/1k3/3/k/5c9296dc6e8c2.png)](http://www.morningbrew.com/)![Advertisement](https://ad.double

In [9]:
split_quiz = scraped_edition.split("<br>")
split_quiz

['[Skip to main content](https://www.morningbrew.com/issues/wexit#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Wexit\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| What the Iran conflict means for gas prices... |\n\n[![](https://media.sailthru.com/5z8/1k3/3/k/5c9296dc6e8c2.png)](http://www.morningbrew.com/)![Advertisement](https://ad.doubl

In [8]:
questions_and_answers = filter_and_clean_parts(scraped_edition)
questions_and_answers

['1. Magma “heartbeat” is [tearing](https://links.morningbrew.com/c/B9G?mblid=0ffe7609f1cc&mbnlid=c2JB5yBiFSSXjqY6QpTPqE) continent apart, geologists discover',
 '2. Board game lovers more likely to have [successful marriages](https://links.morningbrew.com/c/B9H?mblid=8bc17e310b6e&mbnlid=c2JB5yBiFSSXjqY6QpTPqE) than karaoke fans',
 '3. Netflix’s “Liver King” [arrested](https://links.morningbrew.com/c/B9I?mblid=57486c2f40b3&mbnlid=c2JB5yBiFSSXjqY6QpTPqE) after “picking a fight” with Joe Rogan',
 '4. BBC [pledges](https://links.morningbrew.com/c/B9J?mblid=eaf2a5456e6f&mbnlid=c2JB5yBiFSSXjqY6QpTPqE) to be “mindful” of spoilers after irate viewers complain about _Doctor Who_reveal ',
 'We made up the one about board game lovers.']

In [ ]:
url = re.compile(r'(\(.*?\))').search(questions_and_answers[0]).group().strip("()")
questions_and_answers[0].replace(url, "").replace("()", "").replace("[", "").replace("]", "").lstrip("1234. ")

'“Antarctica logs hottest temperature on record with a reading of 18.3C” (65 degrees Fahrenheit)'

In [166]:
# quiz = {
#         "_id": edition_id,
#         "date": date.isoformat(),
#         "source_url": f"https://www.morningbrew.com/issues/{slug}",
#         "headlines": [],
#         "answer": questions_and_answers[4]
#     }

for index in range(4):
    if index < 4:
        headline, url = url_headline_extract(index, questions_and_answers)
        # quiz["headlines"].append({"text": headline, "url": url})
        print(headline, url)#quiz["headlines"].append({"text": headline, "url": url})

# quiz

Jeff Bezos' superyacht is so big it needs its own yacht https://www.cnn.com/2021/05/10/business/jeff-bezos-yacht/index.html
Anti-maskers ready to start masking—to protect themselves from the vaccinated https://www.vice.com/en/article/88nnwg/anti-maskers-ready-to-start-maskingto-protect-themselves-from-the-vaccinated
IHOP turned awayAdam Sandler. Now it's trying to make it up to him https://www.cnn.com/2021/05/07/business/adam-sandler-ihop-response-milkshake-trnd/index.html
LinkedIn pushes to increase users in Florida because residents click most ads https://media0.giphy.com/media/L1EqhrGxoyBBv5s8xG/giphy.gif


In [ ]:
process_edition(index, full_archive_metadata, date, archive, iso_date)

In [16]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
part = '# [Demon Stores Look for New Hosts](https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020?email={{profile.email}}) |\n\n![Spirit Halloween store](https://dlp31coh2a67q.cloudfront.net/eyJrZXkiOiJ1cGxvYWRzL21lZGl1bS9hc3NldC8yOTkwL3NwaXJpdF9oYWxsb3dlZW5fc3RvcmUuanBnIiwiYnVja2V0Ijoib3Nsby1wcm9kdWN0aW9uIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoxMDAwLCJoZWlnaHQiOm51bGx9fX0=)\n\n_Ben McCanna/Portland Portland Press Herald via Getty Images_\n\n|     |\n| --- |\n| Every year, pop-up Halloween stores rise from the underworld to possess empty storefronts vacated by mortal retailers.\n\nTypically, this non-traditional leasing model means seasonal pop-ups perform well when the rest of retail flounders. But even with 2020 laying out a feast of fresh retail corpses, are spooky stores immune to the pandemic? Let\'s check in with two retail reanimaters:\n\n1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+) than last year, and its CEO said last month the company is "anticipating a Halloween on par with last year."\n2. **Party City** was far less bullish, opening 91% fewer of its Halloween City locations this year. With just 25 stores across the U.S., these days it\'s more like a Halloween Town.\n\n**Zoom out:** Halloween City was showing zombie-like symptoms even before the pandemic. Last year, sales per store declined nearly [21% annually](https://www.retaildive.com/news/party-citys-october-sales-a-disaster-with-halloween-stores-down-21/566955/?utm_source=morning_brew)—a major problem for parent Party City considering Halloween sales account for a fifth of its total U.S. retail revenue.\n\n|     |\n| --- |\n| [![](https://media.sailthru.com/5z8/1k4/4/2/5e865734bf6f7.jpg)](https://www.facebook.com/sharer/sharer.php?u=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86574d1eb82.jpg)](https://twitter.com/intent/tweet?text=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year&url=https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020&via=MorningBrew)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86576e04837.jpg)](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020&title=Halloween+Pop-Up+Shops+Have+Mixed+Success+In+2020&source=www.morningbrew.com&summary=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year)[![](https://media.sailthru.com/5z8/1k4/4/2/5e8657887c532.jpg)](mailto:?subject=Check%20out%20this%20story%20from%20Morning%20Brew%21&body=Halloween%20Pop-Up%20Shops%20Have%20Mixed%20Success%20In%202020%3A%20https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020%0A%0AWant%20more%20great%20content%3F%20Subscribe%20to%20Daily%20Brew%20%E2%80%94%20Delivering%20the%20latest%20business%20news%20from%20Wall%20St.%20to%20Silicon%20Valley%2C%20daily.%0Ahttps%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fr%2F%3Fkid%3D{{profile.vars.referral_code}}%26utm_source%3Demail_share%0A) | | |\n| --- |\n\n| |     |\n| --- |\n| ### FOOD'
matched = q_search.search(part)
matched.group()

'1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+)'

In [ ]:
raw_text = '1. "Overdue VHS tape from 1999 leads to [warrant for embezzlement](https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew)"'

url = re.compile(r'\]\((.*?)\)[,.’"”\\\?:]?[”’\']?(?:\s|$)').search(raw_text).group(1)#.strip("](),. ")
url

# urlsplit(url)

'https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew'

In [163]:
raw_text = '1. "Overdue VHS tape from 1999 leads to [warrant for embezzlement](https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew)"'

start = raw_text.find("](") + 2

depth = 1
for i in range(start, len(raw_text)):
    if raw_text[i] == "(":
        depth += 1
    elif raw_text[i] == ")":
        depth -= 1
        if depth == 0:
            end = i
            break

url = raw_text[start:end]
url

'https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew'

In [7]:
with open(f"{data_path}/manual_quiz_additions.json", "r") as f:
    manual_quiz_additions = json.load(f)

print(manual_quiz_additions)

for item in manual_quiz_additions:
    print(item)

new_list = []
print(new_list)
for item in manual_quiz_additions:
    new_list.append(item)

print(new_list)

[{'_id': 'SQTYz1c0U2PEid3hQDOnwo', 'date': '2022-06-30', 'source_url': 'https://www.morningbrew.com/issues/facing-the-music', 'headlines': [{'text': 'Shell is hiring a TikTok manager to reach Gen Z', 'url': 'https://www.bloomberg.com/news/articles/2022-06-29/shell-is-hiring-a-tiktok-channel-manager-to-reach-gen-z?srnd=premium#xj4y7vzkg'}, {'text': 'New Wimbledon documentary has interviews with former pros, journalists, andâ€¦Dick Cheney', 'url': 'https://giphy.com/gifs/wimbledon-dance-tennis-celebrate-Si3yvzzR6HZ1C'}, {'text': 'Analysts accuse Bed Bath & Beyond of turning off AC in stores to save money as sales plummet', 'url': 'https://edition.cnn.com/2022/06/28/investing/bed-bath--beyond-sales-decline/index.html'}, {'text': 'Man loses USB flash drive with data on entire cityâ€™s residents after night out', 'url': 'https://edition.cnn.com/2022/06/24/asia/japan-amagasaki-usb-data-intl-hnk/index.html'}], 'answer': 'We made up the Wimbledon doc one.'}, {'_id': '3wIi2vuFo92PQWFIRJDTrq', '